# Ch.1 — GPU Architecture Fundamentals

**No GPU required.** Every cell in this notebook runs on any laptop using NumPy, Pandas, and Matplotlib. You are building calculators and visualisers — the same tools a Platform Engineer uses to reason about hardware *before* spending money on it.

**What this notebook builds:**
1. A GPU spec database — compare cards side-by-side on the numbers that actually matter
2. A Roofline Model visualiser — see exactly where any operation sits relative to a GPU's limits
3. An arithmetic intensity calculator — compute the FLOP/byte ratio for every major LLM operation
4. A decode step simulator — trace one token generation through the hardware at the byte level
5. A batching curve — watch arithmetic intensity rise as batch size grows
6. An InferenceBase decision tool — which GPU should the startup buy?

In [ ]:
# ── Dependencies ────────────────────────────────────────────────────────────
# All standard. If missing: pip install numpy pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("Libraries loaded. Ready.")

## 1 · GPU Spec Database

The five numbers that matter for AI workloads — everything else is marketing.

| Spec | Why it matters |
|---|---|
| `bf16_tflops` | Peak compute for LLM training/inference (use this, not FP32) |
| `bandwidth_tbs` | HBM bandwidth in TB/s — the real bottleneck for decode |
| `vram_gb` | Maximum model + KV cache you can fit |
| `nvlink_bw_gbs` | Intra-node GPU-to-GPU bandwidth (0 = PCIe only) |
| `tdp_w` | Watts — cooling, rack density, electricity cost |

Run the cell below to load the database, then explore `gpu_db`.

In [ ]:
# ── GPU Spec Database ────────────────────────────────────────────────────────
# All numbers from official NVIDIA datasheets and technical blogs.
# bf16_tflops: BF16 Tensor Core peak (dense, no sparsity)
# bandwidth_tbs: HBM bandwidth in TB/s
# vram_gb: total VRAM
# nvlink_bw_gbs: bidirectional NVLink bandwidth per GPU (0 = PCIe only)
# tdp_w: thermal design power in watts
# approx_price_usd: rough market price as of 2025 (new or used as noted)

gpu_specs = [
    # name                  arch         bf16_tflops  bw_tbs  vram  nvlink  tdp   price   note
    ("H100 SXM",           "Hopper",     989,         3.35,   80,   900,    700,  30000,  "New, DC"),
    ("H100 PCIe",          "Hopper",     378,         2.0,    80,   0,      350,  25000,  "New, DC"),
    ("A100 80GB SXM",      "Ampere",     312,         2.0,    80,   600,    400,  10000,  "Used, DC"),
    ("A100 40GB SXM",      "Ampere",     312,         1.6,    40,   600,    400,  6000,   "Used, DC"),
    ("A100 PCIe",          "Ampere",     312,         1.6,    40,   0,      250,  5000,   "Used, DC"),
    ("A10G",               "Ampere",     125,         0.6,    24,   0,      150,  2500,   "New, DC"),
    ("RTX 4090",           "Ada",        165,         1.008,  24,   0,      450,  1600,   "New, consumer"),
    ("RTX 3090",           "Ampere",     71,          0.936,  24,   0,      350,  700,    "Used, consumer"),
    ("RTX 3080 Ti",        "Ampere",     40,          0.912,  12,   0,      350,  500,    "Used, consumer"),
    ("V100 32GB",          "Volta",      125,         0.9,    32,   300,    300,  1500,   "Used, DC (legacy)"),
]

cols = ["GPU", "Architecture", "BF16_TFLOPS", "Bandwidth_TBs", "VRAM_GB",
        "NVLink_BW_GBs", "TDP_W", "Price_USD", "Notes"]
gpu_db = pd.DataFrame(gpu_specs, columns=cols).set_index("GPU")

# Derived columns
gpu_db["Ridge_Point_FLOPbyte"] = (gpu_db["BF16_TFLOPS"] * 1e12) / (gpu_db["Bandwidth_TBs"] * 1e12)
gpu_db["BW_per_dollar"]        = gpu_db["Bandwidth_TBs"] / gpu_db["Price_USD"] * 1000  # GB/s per $1k
gpu_db["TFLOPS_per_dollar"]    = gpu_db["BF16_TFLOPS"]   / gpu_db["Price_USD"] * 1000  # TFLOPS per $1k

print(gpu_db[["BF16_TFLOPS", "Bandwidth_TBs", "VRAM_GB", "Ridge_Point_FLOPbyte",
              "Price_USD", "TDP_W"]].to_string())

In [ ]:
# ── Bar chart: bandwidth per dollar vs TFLOPS per dollar ────────────────────
# For LLM inference (memory-bound), bandwidth/$ is the key value metric.
# For training (compute-bound at large batch), TFLOPS/$ matters more.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = gpu_db.sort_values("BW_per_dollar", ascending=False).index

ax = axes[0]
bars = ax.barh(order, gpu_db.loc[order, "BW_per_dollar"],
               color=["#2196F3" if "RTX" in g or "A10" in g else "#FF9800" for g in order])
ax.set_xlabel("HBM Bandwidth (GB/s) per $1,000 spent", fontsize=10)
ax.set_title("Inference value: Bandwidth per dollar\n(higher = better for LLM decode)", fontsize=11)
for bar, val in zip(bars, gpu_db.loc[order, "BW_per_dollar"]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}", va="center", fontsize=8)

order2 = gpu_db.sort_values("TFLOPS_per_dollar", ascending=False).index
ax2 = axes[1]
bars2 = ax2.barh(order2, gpu_db.loc[order2, "TFLOPS_per_dollar"],
                 color=["#2196F3" if "RTX" in g or "A10" in g else "#FF9800" for g in order2])
ax2.set_xlabel("BF16 TFLOPS per $1,000 spent", fontsize=10)
ax2.set_title("Training value: TFLOPS per dollar\n(higher = better for batch training)", fontsize=11)

legend = [mpatches.Patch(color="#2196F3", label="Consumer / edge"),
          mpatches.Patch(color="#FF9800", label="Datacenter")]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=9)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

print("\nKey takeaway:")
print("  RTX 4090: best bandwidth-per-dollar — ideal for single-GPU LLM inference on a budget.")
print("  A100/H100: better TFLOPS-per-dollar at scale (large batches, training).")

## 2 · The Roofline Model

The Roofline Model answers one question: *for a given GPU, is my operation limited by compute or by memory?*

$$\text{Achievable TFLOP/s} = \min\!\left(\text{Peak TFLOP/s},\;\; I \times \text{Bandwidth (TB/s)}\right)$$

Where $I$ is **arithmetic intensity** in FLOP/byte:

$$I = \frac{\text{FLOPs performed}}{\text{Bytes read from HBM}}$$

- If $I < I_\text{ridge}$: compute units idle waiting for data → **memory-bound**
- If $I > I_\text{ridge}$: memory system is idle → **compute-bound**
- $I_\text{ridge} = \text{Peak TFLOP/s} \div \text{Bandwidth (TB/s)}$

The `plot_roofline()` function below works for any GPU in `gpu_db`. Pass a list of GPU names and optionally a list of `(label, intensity)` operation points to overlay.

In [ ]:
# ── Roofline Model: reusable plotter ────────────────────────────────────────

def plot_roofline(gpu_names, operations=None, ax=None, title=None):
    """
    Plot the roofline model for one or more GPUs.
    gpu_names  : list of GPU names from gpu_db
    operations : list of (label, arithmetic_intensity_flop_per_byte) to overlay
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    colors = ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0", "#FF9800"]
    intensity_range = np.logspace(-1, 4, 500)   # 0.1 → 10,000 FLOP/byte

    for i, gpu_name in enumerate(gpu_names):
        row = gpu_db.loc[gpu_name]
        peak_tflops  = row["BF16_TFLOPS"]        # TFLOP/s
        bw_tflop_per_byte = row["Bandwidth_TBs"] # TB/s = TFLOP/byte
        ridge = row["Ridge_Point_FLOPbyte"]

        achievable = np.minimum(peak_tflops, intensity_range * bw_tflop_per_byte)
        color = colors[i % len(colors)]
        ax.loglog(intensity_range, achievable, color=color, linewidth=2.5,
                  label=f"{gpu_name} (ridge = {ridge:.0f} FLOP/byte)")
        ax.axvline(ridge, color=color, linestyle=":", linewidth=1, alpha=0.6)
        ax.text(ridge * 1.05, peak_tflops * 0.55, f"{ridge:.0f}",
                color=color, fontsize=8, va="center")

    # Overlay operation points (scored against first GPU)
    if operations:
        row0 = gpu_db.loc[gpu_names[0]]
        op_colors = plt.cm.Set2(np.linspace(0, 1, len(operations)))
        for (label, intensity), oc in zip(operations, op_colors):
            ach = min(row0["BF16_TFLOPS"], intensity * row0["Bandwidth_TBs"])
            ax.scatter(intensity, ach, s=120, color=oc, zorder=5,
                       edgecolors="black", linewidths=0.5)
            ax.annotate(label, (intensity, ach), textcoords="offset points",
                        xytext=(6, 4), fontsize=8, color="black")

    ax.set_xlabel("Arithmetic Intensity (FLOP / byte of HBM traffic)", fontsize=11)
    ax.set_ylabel("Achievable Throughput (TFLOP/s)", fontsize=11)
    ax.set_title(title or "Roofline Model", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, which="both", alpha=0.2)
    ax.text(0.13, ax.get_ylim()[0] * 2.5, "← Memory-bound", fontsize=8, color="gray")
    ax.text(1200, ax.get_ylim()[1] * 0.4, "Compute-bound →", fontsize=8, color="gray")
    return ax


# Compare four GPUs relevant to InferenceBase
fig, ax = plt.subplots(figsize=(12, 7))
plot_roofline(
    ["H100 SXM", "A100 80GB SXM", "RTX 4090", "A10G"],
    ax=ax,
    title="Roofline Model — H100 vs A100 vs RTX 4090 vs A10G (BF16 Tensor)"
)
plt.tight_layout()
plt.show()

print("Ridge points (FLOP/byte) — operations left of the ridge are memory-bound:")
for g in ["H100 SXM", "A100 80GB SXM", "RTX 4090", "A10G"]:
    print(f"  {g:<22}: ridge = {gpu_db.loc[g, 'Ridge_Point_FLOPbyte']:.0f} FLOP/byte")

## 3 · Arithmetic Intensity of LLM Operations

Now let's calculate where real LLM operations land on that roofline.

For a matrix multiply $[B \times M] \times [M \times N]$:

$$\text{FLOPs} = 2 \cdot B \cdot M \cdot N$$
$$\text{Bytes} = (B \cdot M + M \cdot N + B \cdot N) \times \text{bytes per element}$$
$$I = \frac{2BMN}{(BM + MN + BN) \times \text{bpe}}$$

When $B = 1$ (single user, token-by-token decode), the weight matrix $[M \times N]$ completely dominates the byte count. The intensity collapses toward $\approx 1 / \text{bpe}$ — nearly zero relative to the ridge point. This is the root cause of why single-user LLM inference is so GPU-inefficient.

In [ ]:
# ── Arithmetic intensity calculator for matrix multiplies ───────────────────

def gemm_arithmetic_intensity(B, M, N, bytes_per_element=2):
    """
    Arithmetic intensity for [B x M] x [M x N] = [B x N].
    bytes_per_element: 4=FP32, 2=BF16/FP16, 1=INT8, 0.5=INT4/FP8
    """
    flops     = 2 * B * M * N
    bytes_hbm = (B * M + M * N + B * N) * bytes_per_element
    return flops / bytes_hbm


# ── Llama-3-8B model dimensions ─────────────────────────────────────────────
HIDDEN  = 4096
INTER   = 14336   # FFN intermediate (SwiGLU: gate + up projections)
VOCAB   = 32000
SEQ_LEN = 512     # current KV cache length at decode time
BPE     = 2       # bytes per element (BF16)

A100_RIDGE = gpu_db.loc["A100 80GB SXM", "Ridge_Point_FLOPbyte"]

# Operations in one transformer layer, decode phase (batch=1)
ops_b1 = {
    "QKV projection\n[1×4096]×[4096×12288]":  gemm_arithmetic_intensity(1, HIDDEN, HIDDEN*3, BPE),
    "Attn output proj\n[1×4096]×[4096×4096]": gemm_arithmetic_intensity(1, HIDDEN, HIDDEN, BPE),
    "FFN gate+up\n[1×4096]×[4096×14336]":     gemm_arithmetic_intensity(1, HIDDEN, INTER, BPE),
    "FFN down\n[1×14336]×[14336×4096]":        gemm_arithmetic_intensity(1, INTER, HIDDEN, BPE),
    "LM head\n[1×4096]×[4096×32000]":          gemm_arithmetic_intensity(1, HIDDEN, VOCAB, BPE),
    "Attention scores\n[1×128]×[128×512]":     gemm_arithmetic_intensity(1, 128, SEQ_LEN, BPE),
}

print(f"Arithmetic intensity at batch=1 (single-user decode), BF16 | A100 ridge={A100_RIDGE:.0f} FLOP/byte")
print(f"{'Operation':<45} {'FLOP/byte':>12}  Region")
print("-" * 75)
for op, intensity in ops_b1.items():
    label  = op.replace("\n", " ")
    region = "MEMORY-BOUND ←" if intensity < A100_RIDGE else "Compute-bound"
    print(f"{label:<45} {intensity:>10.2f}  {region}")

In [ ]:
# ── Plot all LLM operations on the A100 roofline ────────────────────────────

ops_for_plot = {
    "QKV proj (decode B=1)":       gemm_arithmetic_intensity(1,  HIDDEN, HIDDEN*3, BPE),
    "Attn out (decode B=1)":       gemm_arithmetic_intensity(1,  HIDDEN, HIDDEN,   BPE),
    "FFN gate+up (decode B=1)":    gemm_arithmetic_intensity(1,  HIDDEN, INTER,    BPE),
    "FFN down (decode B=1)":       gemm_arithmetic_intensity(1,  INTER,  HIDDEN,   BPE),
    "LM head (decode B=1)":        gemm_arithmetic_intensity(1,  HIDDEN, VOCAB,    BPE),
    "Attn scores (seq=512)":       gemm_arithmetic_intensity(1,  128,    SEQ_LEN,  BPE),
    "FFN gate+up (prefill B=32)":  gemm_arithmetic_intensity(32, HIDDEN, INTER,    BPE),
    "QKV proj (prefill B=32)":     gemm_arithmetic_intensity(32, HIDDEN, HIDDEN*3, BPE),
    "QKV proj (train B=512)":      gemm_arithmetic_intensity(512,HIDDEN, HIDDEN*3, BPE),
}

decode_ops  = [k for k in ops_for_plot if "decode" in k or "seq=" in k]
prefill_ops = [k for k in ops_for_plot if "prefill" in k]
train_ops   = [k for k in ops_for_plot if "train" in k]

intensity_range = np.logspace(-1, 4, 500)
a100 = gpu_db.loc["A100 80GB SXM"]
peak, bw, ridge = a100["BF16_TFLOPS"], a100["Bandwidth_TBs"], a100["Ridge_Point_FLOPbyte"]

fig, ax = plt.subplots(figsize=(12, 7))
achievable = np.minimum(peak, intensity_range * bw)
ax.loglog(intensity_range, achievable, color="#FF9800", linewidth=2.5,
          label=f"A100 80GB roofline (ridge = {ridge:.0f} FLOP/byte)")
ax.axvline(ridge, color="#FF9800", linestyle="--", linewidth=1)
ax.text(ridge * 1.08, 1.5, f"Ridge\n{ridge:.0f}", color="#FF9800", fontsize=8)

for op_name, intensity in ops_for_plot.items():
    ach = min(peak, intensity * bw)
    if op_name in decode_ops:
        color, marker = "#F44336", "v"
    elif op_name in prefill_ops:
        color, marker = "#2196F3", "^"
    else:
        color, marker = "#4CAF50", "D"
    ax.scatter(intensity, ach, s=140, color=color, zorder=5,
               edgecolors="black", linewidths=0.5, marker=marker)
    ax.annotate(op_name, (intensity, ach), textcoords="offset points",
                xytext=(6, 4), fontsize=7.5, color=color)

legend_handles = [
    Line2D([0],[0], marker="v", color="w", markerfacecolor="#F44336", markersize=10,
           label="Decode phase (batch=1)"),
    Line2D([0],[0], marker="^", color="w", markerfacecolor="#2196F3", markersize=10,
           label="Prefill phase (batch=32)"),
    Line2D([0],[0], marker="D", color="w", markerfacecolor="#4CAF50", markersize=10,
           label="Training (batch=512)"),
    Line2D([0],[0], color="#FF9800", linewidth=2, label="A100 80GB roofline"),
]
ax.legend(handles=legend_handles, fontsize=9, loc="upper left")
ax.axvspan(0.1, ridge, alpha=0.05, color="red")
ax.text(0.13, 0.2, "Memory-bound zone", color="#F44336", fontsize=9, alpha=0.7)
ax.set_xlabel("Arithmetic Intensity (FLOP / byte)", fontsize=11)
ax.set_ylabel("Achievable Throughput (TFLOP/s)", fontsize=11)
ax.set_title("A100 Roofline — Llama-3-8B operations: decode vs prefill vs training batch",
             fontsize=12, fontweight="bold")
ax.grid(True, which="both", alpha=0.2)
plt.tight_layout()
plt.show()

print("All decode ops (red) cluster far left — deep memory-bound.")
print("Training at batch=512 (green) approaches or crosses the ridge — compute starts to matter.")

## 4 · Decode Step Simulator — One Token at the Hardware Level

Let's trace exactly what happens during a single decode step for Llama-3-8B: FLOPs performed, bytes moved from HBM, and the resulting minimum latency bounded by memory bandwidth — not compute.

The simulator accounts for every operation in every layer: QKV projections, attention score computation, softmax + weighted sum, output projection, and both FFN linear layers. The LM head (vocab projection) is included once.

In [ ]:
# ── Full decode step: FLOP and HBM byte accounting for Llama-3-8B ───────────

def decode_step_analysis(
    batch_size = 1,
    hidden_dim = 4096,
    num_layers = 32,
    num_heads  = 32,
    head_dim   = 128,
    ffn_dim    = 14336,
    vocab_size = 32000,
    seq_len    = 512,
    bpe        = 2,       # bytes per element (BF16)
):
    """
    Returns FLOPs, HBM bytes, and arithmetic intensity for one decode step.
    Covers all transformer layers + the LM head.
    """
    B = batch_size
    results = []

    for layer in range(num_layers):
        # QKV: [B, hidden] x [hidden, 3*hidden]
        qkv_flops = 2 * B * hidden_dim * (3 * hidden_dim)
        qkv_bytes = (B * hidden_dim + hidden_dim * (3*hidden_dim) + B * (3*hidden_dim)) * bpe

        # Attention scores: [B*H, 1, D] x [B*H, D, seq_len]
        attn_flops = 2 * B * num_heads * 1 * head_dim * seq_len
        attn_bytes = (B * num_heads * 1 * head_dim + B * num_heads * head_dim * seq_len
                      + B * num_heads * 1 * seq_len) * bpe

        # Softmax + weighted sum (same shape as scores pass)
        sv_flops = 2 * B * num_heads * 1 * seq_len * head_dim
        sv_bytes = attn_bytes

        # Output projection: [B, hidden] x [hidden, hidden]
        out_flops = 2 * B * hidden_dim * hidden_dim
        out_bytes = (B * hidden_dim + hidden_dim * hidden_dim + B * hidden_dim) * bpe

        # FFN gate + up: [B, hidden] x [hidden, ffn_dim]  (×2 for both projections)
        ffn1_flops = 2 * 2 * B * hidden_dim * ffn_dim
        ffn1_bytes = (B * hidden_dim + hidden_dim * ffn_dim * 2 + B * ffn_dim * 2) * bpe

        # FFN down: [B, ffn_dim] x [ffn_dim, hidden]
        ffn2_flops = 2 * B * ffn_dim * hidden_dim
        ffn2_bytes = (B * ffn_dim + ffn_dim * hidden_dim + B * hidden_dim) * bpe

        results.append({
            "layer": layer,
            "qkv_flops":  qkv_flops,  "qkv_bytes":  qkv_bytes,
            "attn_flops": attn_flops, "attn_bytes": attn_bytes,
            "sv_flops":   sv_flops,   "sv_bytes":   sv_bytes,
            "out_flops":  out_flops,  "out_bytes":  out_bytes,
            "ffn1_flops": ffn1_flops, "ffn1_bytes": ffn1_bytes,
            "ffn2_flops": ffn2_flops, "ffn2_bytes": ffn2_bytes,
        })

    df = pd.DataFrame(results)
    lm_flops = 2 * B * hidden_dim * vocab_size
    lm_bytes = (B * hidden_dim + hidden_dim * vocab_size + B * vocab_size) * bpe

    total_flops = df.filter(like="_flops").values.sum() + lm_flops
    total_bytes = df.filter(like="_bytes").values.sum() + lm_bytes

    return {
        "total_flops": total_flops,
        "total_bytes": total_bytes,
        "arithmetic_intensity": total_flops / total_bytes,
        "per_layer_df": df,
        "lm_flops": lm_flops,
        "lm_bytes": lm_bytes,
    }


# ── Single-user decode analysis ──────────────────────────────────────────────
OVERHEAD = 1.5   # practical vs theoretical: kernel launch, allocation, scheduling

result_b1 = decode_step_analysis(batch_size=1)
print("Llama-3-8B — one decode step | batch=1, seq_len=512, BF16")
print(f"  Total FLOPs:          {result_b1['total_flops'] / 1e9:.2f} GFLOPs")
print(f"  Total HBM bytes:      {result_b1['total_bytes'] / 1e9:.2f} GB")
print(f"  Arithmetic intensity: {result_b1['arithmetic_intensity']:.3f} FLOP/byte")
print()

# Minimum decode time on each GPU (bandwidth-limited)
print(f"{'GPU':<22} {'BW (TB/s)':>10} {'Min step (ms)':>15} {'Max tok/s':>12}")
print("-" * 65)
for g in ["H100 SXM", "A100 80GB SXM", "A100 PCIe", "RTX 4090", "A10G"]:
    bw_bps = gpu_db.loc[g, "Bandwidth_TBs"] * 1e12
    t_ms   = result_b1["total_bytes"] / bw_bps * 1000 * OVERHEAD
    toks   = 1000 / t_ms
    print(f"{g:<22} {gpu_db.loc[g,'Bandwidth_TBs']:>10.3f} {t_ms:>15.2f} {toks:>12.0f}")

print()
print("Lower bound: assumes 100% bandwidth utilisation. Actual: ~1.3–2× slower.")

In [ ]:
# ── Per-operation breakdown: FLOPs vs HBM bytes in one layer ────────────────

df = result_b1["per_layer_df"]
op_labels = ["QKV proj", "Attn scores", "Softmax+SV", "Attn output", "FFN gate+up", "FFN down"]
flop_cols  = ["qkv_flops", "attn_flops", "sv_flops", "out_flops", "ffn1_flops", "ffn2_flops"]
byte_cols  = ["qkv_bytes", "attn_bytes", "sv_bytes", "out_bytes", "ffn1_bytes", "ffn2_bytes"]

avg_flops = [df[c].mean() / 1e6 for c in flop_cols]   # MFLOPs
avg_bytes = [df[c].mean() / 1e6 for c in byte_cols]   # MB

x = np.arange(len(op_labels))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bars1 = ax1.bar(x, avg_flops, color="#2196F3", alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels(op_labels, rotation=25, ha="right", fontsize=9)
ax1.set_ylabel("MFLOPs per layer (batch=1)")
ax1.set_title("Compute per operation")
for b in bars1:
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
             f"{b.get_height():.0f}", ha="center", fontsize=8)

bars2 = ax2.bar(x, avg_bytes, color="#FF5722", alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(op_labels, rotation=25, ha="right", fontsize=9)
ax2.set_ylabel("MB read/written per layer (batch=1)")
ax2.set_title("HBM memory traffic per operation")
for b in bars2:
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 0.2,
             f"{b.get_height():.0f}", ha="center", fontsize=8)

fig.suptitle("Llama-3-8B — FLOPs vs HBM traffic per operation (one layer, batch=1)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Weight matrices (QKV, FFN) dominate HBM traffic: loaded fully from VRAM even")
print("when computing just 1 token. The FLOPs are minimal by comparison — that mismatch")
print("is why batch_size=1 wastes most of the GPU's compute capacity.")

## 5 · The Batching Curve — Why Batching Transforms Efficiency

When processing $B$ tokens simultaneously instead of 1, the weight matrices are still the same size (loaded once from HBM), but you perform $B\times$ more FLOPs on them. Arithmetic intensity rises linearly with batch size — until you cross the ridge point and become compute-bound.

$$I(B) \xrightarrow{B \gg 1} \frac{2B}{\text{bpe}}$$

This is the fundamental reason why **continuous batching** (Ch.5) is the single most impactful inference optimisation: it raises the effective batch size from ~1 toward 32–128 without adding latency per user.

In [ ]:
# ── Batch size sweep: intensity and GPU utilisation on A100 ─────────────────

batch_sizes = np.arange(1, 257)
intensities = np.array([
    decode_step_analysis(batch_size=int(b))["arithmetic_intensity"]
    for b in batch_sizes
])

a100  = gpu_db.loc["A100 80GB SXM"]
ridge = a100["Ridge_Point_FLOPbyte"]
peak  = a100["BF16_TFLOPS"]
bw    = a100["Bandwidth_TBs"]

achievable  = np.minimum(peak, intensities * bw)
utilisation = (achievable / peak) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: arithmetic intensity vs batch size
ax1.semilogx(batch_sizes, intensities, color="#2196F3", linewidth=2.5, label="Arithmetic intensity")
ax1.axhline(ridge, color="#FF9800", linestyle="--", linewidth=1.5,
            label=f"A100 ridge ({ridge:.0f} FLOP/byte)")
ax1.fill_between(batch_sizes, 0, np.minimum(intensities, ridge),
                 alpha=0.1, color="red", label="Memory-bound")
ax1.fill_between(batch_sizes, np.minimum(intensities, ridge), intensities,
                 alpha=0.1, color="green", label="Compute-bound")
ax1.set_xlabel("Batch size (log scale)")
ax1.set_ylabel("Arithmetic Intensity (FLOP/byte)")
ax1.set_title("Arithmetic intensity rises with batch size\n(Llama-3-8B, BF16, A100)")
ax1.legend(fontsize=9)
ax1.set_xlim(1, 256)

cross = batch_sizes[intensities >= ridge]
if len(cross):
    cx = cross[0]
    ax1.axvline(cx, color="#4CAF50", linestyle=":", linewidth=1.5)
    ax1.text(cx * 1.1, ridge * 0.65, f"Batch={cx}\ncrosses\nridge", fontsize=8, color="#4CAF50")

# Plot 2: compute utilisation
ax2.semilogx(batch_sizes, utilisation, color="#4CAF50", linewidth=2.5)
ax2.axhline(100, color="#FF9800", linestyle="--", linewidth=1, label="100% utilisation")
ax2.set_xlabel("Batch size (log scale)")
ax2.set_ylabel("Compute Utilisation (% of peak BF16 TFLOPS)")
ax2.set_title("GPU compute utilisation vs batch size\n(A100 80GB, Llama-3-8B decode)")
ax2.set_ylim(0, 110)
ax2.set_xlim(1, 256)
ax2.legend(fontsize=9)

# Annotate key batch sizes
for bs in [1, 4, 16, 32, 64, 128]:
    idx  = bs - 1
    util = utilisation[idx]
    ax2.scatter(bs, util, s=80, color="#2196F3", zorder=5)
    ax2.annotate(f"B={bs}\n{util:.0f}%", (bs, util),
                 textcoords="offset points", xytext=(8, -20),
                 fontsize=8, color="#2196F3",
                 arrowprops=dict(arrowstyle="-", color="#2196F3", lw=0.7))

plt.tight_layout()
plt.show()

print("Utilisation numbers (A100 80GB, Llama-3-8B decode):")
for bs in [1, 4, 8, 16, 32, 64, 128]:
    print(f"  batch={bs:>3}: intensity={intensities[bs-1]:.2f}  utilisation={utilisation[bs-1]:.1f}%")

## 6 · InferenceBase Decision Tool

The Platform Engineer needs to answer: **which GPU should InferenceBase buy to serve Llama-3-8B at 5,000 tokens/second total (100 concurrent users × 50 tok/s each), within a $15,000/month cloud budget?**

We model this using only bandwidth arithmetic and published cloud pricing — no benchmarks, no live GPU required.

In [ ]:
# ── InferenceBase: GPU selection model ──────────────────────────────────────

TARGET_TOKS_PER_SEC = 5000   # 100 users × 50 tok/s
SERVE_BATCH         = 32     # realistic vLLM continuous batching batch size
OVERHEAD            = 1.5    # practical vs theoretical throughput

# On-demand hourly prices (approximate 2025 rates)
cloud_instances = {
    "H100 SXM":      3.00,
    "A100 80GB SXM": 2.00,
    "A100 PCIe":     1.75,
    "RTX 4090":      0.80,
    "A10G":          1.00,
}

res_b32       = decode_step_analysis(batch_size=SERVE_BATCH)
bytes_per_step = res_b32["total_bytes"]
llama8b_vram  = 8e9 * 2 / 1e9   # 16 GB for weights alone (BF16)

print(f"Serving config: batch={SERVE_BATCH}, target={TARGET_TOKS_PER_SEC} tok/s total")
print(f"HBM bytes per step (batch={SERVE_BATCH}): {bytes_per_step/1e9:.2f} GB\n")

rows = []
header = f"{'GPU':<22} {'BW TB/s':>8} {'VRAM GB':>8} {'tok/s/GPU':>11} {'GPUs':>6} {'$/month':>10} {'Fits BF16?':>11}"
print(header); print("-" * 82)

for gpu_name, hourly in cloud_instances.items():
    row      = gpu_db.loc[gpu_name]
    bw_bps   = row["Bandwidth_TBs"] * 1e12
    step_t   = (bytes_per_step / bw_bps) * OVERHEAD
    toks_gpu = SERVE_BATCH / step_t
    n_gpus   = int(np.ceil(TARGET_TOKS_PER_SEC / toks_gpu))
    cost_mo  = n_gpus * hourly * 24 * 30
    fits     = "✓ Yes" if row["VRAM_GB"] >= llama8b_vram + 2 else "⚠ Tight"

    rows.append({"GPU": gpu_name, "tok/s/GPU": toks_gpu, "GPUs": n_gpus,
                 "Cost/mo": cost_mo, "Fits": fits, "hourly": hourly})
    print(f"{gpu_name:<22} {row['Bandwidth_TBs']:>8.3f} {row['VRAM_GB']:>8.0f} "
          f"{toks_gpu:>11.0f} {n_gpus:>6} ${cost_mo:>9,.0f} {fits:>11}")

print(f"\nBudget: $15,000/month | Llama-3-8B BF16 weight footprint: ~{llama8b_vram:.0f} GB (need ~18–22 GB total)")

In [ ]:
# ── Visualise cost and throughput comparison ─────────────────────────────────

results_df = pd.DataFrame(rows).set_index("GPU")
gpus   = results_df.index.tolist()
costs  = results_df["Cost/mo"].values
toks   = results_df["tok/s/GPU"].values
ngpus  = results_df["GPUs"].values
BUDGET = 15_000

bar_colors = ["#4CAF50" if c <= BUDGET else "#FF9800" if c <= BUDGET * 2 else "#F44336"
              for c in costs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bars1 = ax1.barh(gpus, costs, color=bar_colors, alpha=0.85)
ax1.axvline(BUDGET, color="black", linestyle="--", linewidth=1.5, label=f"${BUDGET:,} budget")
ax1.set_xlabel("Estimated 30-day cost (USD, on-demand)")
ax1.set_title("Monthly cost to hit 5,000 tok/s\n(InferenceBase, Llama-3-8B BF16)")
ax1.legend()
for bar, val, n in zip(bars1, costs, ngpus):
    ax1.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
             f"${val:,.0f}  ×{n}", va="center", fontsize=8)

bars2 = ax2.barh(gpus, toks, color="#2196F3", alpha=0.85)
ax2.set_xlabel("Tokens/second per single GPU (batch=32)")
ax2.set_title("Single-GPU throughput\n(Llama-3-8B decode, batch=32)")
for bar, val in zip(bars2, toks):
    ax2.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             f"{val:.0f} tok/s", va="center", fontsize=8)

plt.tight_layout()
plt.show()

# Verdict
in_budget = results_df[results_df["Cost/mo"] <= BUDGET].sort_values("Cost/mo")
print("Options within $15k/month budget (on-demand):")
if len(in_budget):
    print(in_budget[["tok/s/GPU", "GPUs", "Cost/mo", "Fits"]].to_string())
    print("\nNote: Spot/reserved pricing (Ch.8) cuts cost by 60–80% — RTX 4090 spot")
    print("instances make more options viable within budget.")
else:
    print("  None fit on-demand — use spot/reserved pricing (see Ch.8).")

## 7 · Sensitivity Analysis — How Model Size Changes Everything

How do the key metrics change as the model scales from 1B to 70B parameters? The VRAM footprint is the hard constraint. Arithmetic intensity at batch=1 barely changes with model size — all models are equally memory-bound per token. But the bandwidth requirement (HBM bytes per decode step) scales directly with parameter count.

In [ ]:
# ── Model size sweep: VRAM, HBM traffic, arithmetic intensity ───────────────

model_configs = {
    "1B":  dict(hidden=2048, layers=16, heads=16, head_dim=128, ffn=8192,  params_b=1),
    "3B":  dict(hidden=3072, layers=28, heads=24, head_dim=128, ffn=8192,  params_b=3),
    "8B":  dict(hidden=4096, layers=32, heads=32, head_dim=128, ffn=14336, params_b=8),
    "13B": dict(hidden=5120, layers=40, heads=40, head_dim=128, ffn=13824, params_b=13),
    "34B": dict(hidden=7168, layers=48, heads=56, head_dim=128, ffn=20480, params_b=34),
    "70B": dict(hidden=8192, layers=80, heads=64, head_dim=128, ffn=28672, params_b=70),
}

sweep_rows = []
for label, cfg in model_configs.items():
    res = decode_step_analysis(
        batch_size=1, hidden_dim=cfg["hidden"], num_layers=cfg["layers"],
        num_heads=cfg["heads"], head_dim=cfg["head_dim"], ffn_dim=cfg["ffn"],
        vocab_size=32000, seq_len=512, bpe=2,
    )
    sweep_rows.append({
        "Model":              label,
        "Params (B)":         cfg["params_b"],
        "VRAM BF16 (GB)":     cfg["params_b"] * 2,
        "HBM bytes/step (GB)":res["total_bytes"] / 1e9,
        "Arithmetic I (B=1)": res["arithmetic_intensity"],
        "GFLOPs/step":        res["total_flops"] / 1e9,
    })

sweep_df = pd.DataFrame(sweep_rows).set_index("Model")
print(sweep_df.to_string(float_format=lambda x: f"{x:.3f}"))
print()
print("Note: arithmetic intensity at batch=1 barely varies — all models are equally memory-bound.")
print("VRAM capacity is the true hard constraint: 70B in BF16 = 140 GB (needs 2× A100 80GB).")

In [ ]:
# ── Plot model size sweep ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
labels = sweep_df.index.tolist()

# 1. VRAM footprint
ax = axes[0]
bars = ax.bar(labels, sweep_df["VRAM BF16 (GB)"], color="#2196F3", alpha=0.85)
for gpu_label, gpu_color, ls in [("A10G", "#FF5722", "--"), ("A100 80GB SXM", "#4CAF50", "-.")]:
    cap = gpu_db.loc[gpu_label, "VRAM_GB"]
    ax.axhline(cap, color=gpu_color, linestyle=ls, linewidth=1.5,
               label=f"{gpu_label} ({cap:.0f} GB)")
for b, v in zip(bars, sweep_df["VRAM BF16 (GB)"]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1, f"{v:.0f} GB",
            ha="center", fontsize=8)
ax.set_ylabel("VRAM (GB, weights only, BF16)")
ax.set_title("Model VRAM footprint")
ax.legend(fontsize=8)

# 2. Arithmetic intensity (barely varies)
ax2 = axes[1]
ax2.bar(labels, sweep_df["Arithmetic I (B=1)"], color="#FF5722", alpha=0.85)
ax2.axhline(gpu_db.loc["A100 80GB SXM", "Ridge_Point_FLOPbyte"],
            color="#FF9800", linestyle="--", linewidth=1.5, label="A100 ridge (156)")
ax2.set_ylabel("Arithmetic Intensity (FLOP/byte, batch=1)")
ax2.set_title("All models equally memory-bound\nat batch=1")
ax2.legend(fontsize=9)
ax2.set_ylim(0, 20)

# 3. HBM bytes per decode step
ax3 = axes[2]
bars3 = ax3.bar(labels, sweep_df["HBM bytes/step (GB)"], color="#9C27B0", alpha=0.85)
for b in bars3:
    ax3.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
             f"{b.get_height():.1f}", ha="center", fontsize=8)
ax3.set_ylabel("HBM bytes per decode step (GB)")
ax3.set_title("Memory traffic scales linearly\nwith model size")

fig.suptitle("How model size affects VRAM, arithmetic intensity, and memory traffic",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## Exercises

Work through these before moving to Ch.2. Each one extends the calculators above.

**Exercise 1** — Add your own GPU to `gpu_specs` in section 1 and re-run all cells. Where does it sit on the roofline?

**Exercise 2** — FP8 inference on H100. Re-run `decode_step_analysis` with `bpe=0.5`. Does the H100's higher ridge point make FP8 decode compute-bound before BF16?

**Exercise 3** — Longer context. Re-run `decode_step_analysis` with `seq_len=4096`. Does quadrupling the context significantly change arithmetic intensity?

**Exercise 4** — The 70B upgrade. Model what happens if InferenceBase upgrades to Llama-3-70B. Does any single GPU fit the model in BF16? What is the minimum GPU count and cost?

In [ ]:
# Exercise 1 — Add your GPU
# Fill in specs, then re-run from Cell 3 onward.

# YOUR_GPU = ("RTX 4080", "Ada", 122, 0.717, 16, 0, 320, 1000, "Consumer")
# Append to gpu_specs and re-run the database cell.
print("Uncomment YOUR_GPU above with your card's specs and re-run from the GPU database cell.")

In [ ]:
# Exercise 2 — FP8 on H100 (0.5 bytes/element instead of 2)
result_fp8  = decode_step_analysis(batch_size=1, bpe=0.5)
result_bf16 = decode_step_analysis(batch_size=1, bpe=2.0)
h100_ridge  = gpu_db.loc["H100 SXM", "Ridge_Point_FLOPbyte"]

print(f"H100 SXM ridge: {h100_ridge:.0f} FLOP/byte")
print(f"BF16 decode intensity: {result_bf16['arithmetic_intensity']:.3f}")
print(f"FP8  decode intensity: {result_fp8['arithmetic_intensity']:.3f}")
print()

# Batch size sweep comparison
b_arr      = np.arange(1, 129)
i_bf16 = np.array([decode_step_analysis(int(b), bpe=2.0)["arithmetic_intensity"] for b in b_arr])
i_fp8  = np.array([decode_step_analysis(int(b), bpe=0.5)["arithmetic_intensity"] for b in b_arr])

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(b_arr, i_bf16, label="BF16 (2 bpe)", color="#2196F3", lw=2)
ax.semilogx(b_arr, i_fp8,  label="FP8  (0.5 bpe)", color="#4CAF50", lw=2)
ax.axhline(h100_ridge, color="#FF9800", linestyle="--", lw=1.5, label=f"H100 ridge ({h100_ridge:.0f})")
ax.set_xlabel("Batch size"); ax.set_ylabel("Arithmetic Intensity (FLOP/byte)")
ax.set_title("FP8 vs BF16: intensity vs batch size (H100, Llama-3-8B)")
ax.legend(); ax.grid(True, which="both", alpha=0.2)
plt.tight_layout(); plt.show()

cross_bf16 = b_arr[i_bf16 >= h100_ridge]
cross_fp8  = b_arr[i_fp8  >= h100_ridge]
print(f"BF16 crosses H100 ridge at batch ≈ {cross_bf16[0] if len(cross_bf16) else '>128'}")
print(f"FP8  crosses H100 ridge at batch ≈ {cross_fp8[0]  if len(cross_fp8)  else '>128'}")

In [ ]:
# Exercise 3 — Longer context (seq_len=4096 vs 512)
short = decode_step_analysis(batch_size=1, seq_len=512)
long  = decode_step_analysis(batch_size=1, seq_len=4096)

print("Effect of longer context (batch=1):")
print(f"  seq_len=512:   intensity={short['arithmetic_intensity']:.4f}  HBM={short['total_bytes']/1e9:.2f} GB")
print(f"  seq_len=4096:  intensity={long['arithmetic_intensity']:.4f}  HBM={long['total_bytes']/1e9:.2f} GB")
print()
print("Long context adds attention-score memory traffic but weights dominate at batch=1.")
print("Arithmetic intensity barely moves — both are equally memory-bound.")

In [ ]:
# Exercise 4 — The 70B upgrade decision
res_70b = decode_step_analysis(
    batch_size=SERVE_BATCH, hidden_dim=8192, num_layers=80, num_heads=64,
    head_dim=128, ffn_dim=28672, vocab_size=32000, seq_len=512, bpe=2
)

print("Llama-3-70B decode analysis (batch=32, BF16):")
print(f"  HBM bytes per step: {res_70b['total_bytes']/1e9:.2f} GB")
print(f"  Arithmetic intensity: {res_70b['arithmetic_intensity']:.3f} FLOP/byte")
print(f"  Weight VRAM: {70*2:.0f} GB → needs 2× A100 80GB or 1× H100 80GB\n")

print(f"{'GPU':<22} {'tok/s/GPU':>10} {'GPUs':>6} {'$/month':>10}  Note")
print("-" * 70)
for gpu_name, hourly in cloud_instances.items():
    row     = gpu_db.loc[gpu_name]
    bw_bps  = row["Bandwidth_TBs"] * 1e12
    step_t  = (res_70b["total_bytes"] / bw_bps) * OVERHEAD
    toks    = SERVE_BATCH / step_t
    n       = int(np.ceil(TARGET_TOKS_PER_SEC / toks))
    cost    = n * hourly * 24 * 30
    single  = "single card can't fit 70B BF16" if row["VRAM_GB"] < 140 else ""
    print(f"{gpu_name:<22} {toks:>10.0f} {n:>6} ${cost:>9,.0f}  {single}")